In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_json, struct
)
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_fact_incremental_" + str(datetime.date.today())

# Source silver tables
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_sales_order_lines = "retail.silver.ns_sales_order_lines"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"
gold_fact_sales = "retail.gold.fact_sales"


# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = CAST({rows_read} AS BIGINT),
          rows_written = CAST({rows_written} AS BIGINT),
          rows_rejected = CAST({rows_rejected} AS BIGINT),
          records_before = CAST({records_before} AS BIGINT),
          records_after = CAST({records_after} AS BIGINT)
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = str(error_message).replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta")
        .mode("append")
        .saveAsTable(reject_table)
    )


def get_unprocessed_silver_run_ids(source_table, gold_table_name):
    return [
        r.etl_run_id
        for r in spark.sql(f"""
            SELECT DISTINCT s.etl_run_id
            FROM {source_table} s
            LEFT ANTI JOIN {processed_runs_table} p
              ON s.etl_run_id = p.source_run_id
             AND p.table_name = '{gold_table_name}'
             AND p.source_layer = 'silver'
             AND p.target_layer = 'gold'
             AND p.status = 'SUCCESS'
            WHERE s.etl_run_id IS NOT NULL
        """).collect()
    ]


def log_layer_processed_runs_gold(source_run_ids, target_run_id, table_name, rows_read, rows_written, rows_rejected):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


# =========================================================
# FACT SALES INCREMENTAL
# =========================================================
def load_fact_sales_incremental():
    table_name = "fact_sales"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(gold_fact_sales).count()

        sol_run_ids = get_unprocessed_silver_run_ids(silver_sales_order_lines, table_name)
        so_run_ids = get_unprocessed_silver_run_ids(silver_sales_orders, table_name)
        sh_run_ids = get_unprocessed_silver_run_ids(silver_shipments, table_name)

        source_run_ids = list(set(sol_run_ids + so_run_ids + sh_run_ids))

        if not source_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_fact_sales}: no new silver runs")
            return

        # =========================================================
        # 1. Find affected sales orders
        # =========================================================
        affected_from_lines = (
            spark.table(silver_sales_order_lines)
            .filter(col("etl_run_id").isin(sol_run_ids))
            .select(col("sales_order_internal_id").alias("sales_order_internal_id"))
        )

        affected_from_orders = (
            spark.table(silver_sales_orders)
            .filter(col("etl_run_id").isin(so_run_ids))
            .select(col("internal_id").alias("sales_order_internal_id"))
        )

        affected_from_shipments = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("sales_order_internal_id").alias("sales_order_internal_id"))
        )

        df_affected_orders = (
            affected_from_lines
            .union(affected_from_orders)
            .union(affected_from_shipments)
            .filter(col("sales_order_internal_id").isNotNull())
            .dropDuplicates()
        )

        affected_order_count = df_affected_orders.count()

        if affected_order_count == 0:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_fact_sales}: no affected orders found")
            return

        # =========================================================
        # 2. Read affected sales order lines
        # =========================================================
        df_sol = (
            spark.table(silver_sales_order_lines)
            .join(
                df_affected_orders,
                on="sales_order_internal_id",
                how="inner"
            )
            .select(
                col("internal_id").alias("sol_internal_id"),
                col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
                col("line_num").alias("sol_line_num"),
                col("item_internal_id").alias("sol_item_internal_id"),
                col("quantity").alias("sol_quantity"),
                col("unit_rate").alias("sol_unit_rate"),
                col("line_amount").alias("sol_line_amount"),
                col("record_hash").alias("sol_record_hash"),
                col("created_at").alias("sol_created_at")
            )
        )

        df_so = (
            spark.table(silver_sales_orders)
            .select(
                col("internal_id").alias("so_internal_id"),
                col("customer_internal_id").alias("so_customer_internal_id"),
                col("order_date").alias("so_order_date"),
                col("order_status").alias("so_order_status"),
                col("sales_channel").alias("so_sales_channel"),
                col("payment_status").alias("so_payment_status"),
                col("currency_code").alias("so_currency_code"),
                col("integration_source").alias("so_integration_source"),
                col("order_total").alias("so_order_total"),
                col("source_system").alias("so_source_system"),
                col("etl_run_id").alias("so_etl_run_id"),
                col("record_hash").alias("so_record_hash")
            )
        )

        df_sh = (
            spark.table(silver_shipments)
            .select(
                col("internal_id").alias("sh_internal_id"),
                col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
                col("shipment_date").alias("sh_shipment_date"),
                col("delivery_date").alias("sh_delivery_date"),
                col("shipment_status").alias("sh_shipment_status"),
                col("warehouse_name").alias("sh_warehouse_name"),
                col("tracking_number").alias("sh_tracking_number"),
                col("record_hash").alias("sh_record_hash")
            )
        )

        df_joined = (
            df_sol
            .join(
                df_so,
                col("sol_sales_order_internal_id") == col("so_internal_id"),
                "inner"
            )
            .join(
                df_sh,
                col("so_internal_id") == col("sh_sales_order_internal_id"),
                "left"
            )
        )

        rows_read = df_joined.count()

        # =========================================================
        # 3. Dimension lookups
        # =========================================================
        df_customer_lkp = (
            spark.table(gold_dim_customer)
            .filter(col("is_current") == True)
            .select(
                col("customer_key"),
                col("customer_internal_id").cast("int").alias("lkp_customer_internal_id")
            )
        )

        df_item_lkp = (
            spark.table(gold_dim_item)
            .filter(col("is_current") == True)
            .select(
                col("item_key"),
                col("item_internal_id").cast("int").alias("lkp_item_internal_id")
            )
        )

        df_date_lkp = spark.table(gold_dim_date).select(
            col("date_key"),
            col("full_date").alias("lkp_full_date")
        )

        df_fact_pre = (
            df_joined
            .join(
                df_customer_lkp,
                col("so_customer_internal_id").cast("int") == col("lkp_customer_internal_id"),
                "left"
            )
            .join(
                df_item_lkp,
                col("sol_item_internal_id").cast("int") == col("lkp_item_internal_id"),
                "left"
            )
            .join(
                df_date_lkp,
                F.to_date(col("so_order_date")) == col("lkp_full_date"),
                "left"
            )
        )

        # =========================================================
        # 4. Handle failed lookups using Unknown Dimension Strategy
        # =========================================================
        unknown_customer_key = spark.sql("""
            SELECT customer_key
            FROM retail.gold.dim_customer
            WHERE customer_internal_id = -1
            AND is_current = true
        """).collect()[0]["customer_key"]

        unknown_item_key = spark.sql("""
            SELECT item_key
            FROM retail.gold.dim_item
            WHERE item_internal_id = -1
            AND is_current = true
        """).collect()[0]["item_key"]

        unknown_date_key = -1

        df_fact_pre = (
            df_fact_pre
            .withColumn(
                "customer_key",
                F.coalesce(col("customer_key"), lit(unknown_customer_key).cast("bigint"))
            )
            .withColumn(
                "item_key",
                F.coalesce(col("item_key"), lit(unknown_item_key).cast("bigint"))
            )
            .withColumn(
                "date_key",
                F.coalesce(col("date_key"), lit(unknown_date_key).cast("int"))
            )
        )

        rows_rejected = 0
        df_valid = df_fact_pre

        # =========================================================
        # 5. Prepare fact table rows
        # =========================================================
        df_tgt = (
            df_valid
            .select(
                col("sol_internal_id").alias("sales_order_line_internal_id"),
                col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
                col("sol_line_num").alias("sales_order_line_num"),
                col("date_key").alias("order_date_key"),
                col("customer_key"),
                col("item_key"),
                col("sol_quantity").alias("quantity"),
                col("sol_unit_rate").alias("unit_rate"),
                col("sol_line_amount").alias("line_amount"),
                col("so_order_total").alias("order_total"),
                col("so_order_status").alias("order_status"),
                col("so_sales_channel").alias("sales_channel"),
                col("so_payment_status").alias("payment_status"),
                col("so_currency_code").alias("currency_code"),
                col("so_integration_source").alias("integration_source"),
                col("sh_shipment_date").alias("shipment_date"),
                col("sh_delivery_date").alias("delivery_date"),
                col("sh_shipment_status").alias("shipment_status"),
                col("sh_warehouse_name").alias("warehouse_name"),
                col("sh_tracking_number").alias("tracking_number"),
                col("so_source_system").alias("source_system"),
                col("so_etl_run_id").alias("etl_run_id"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(col("sol_record_hash"), lit("")),
                        F.coalesce(col("so_record_hash"), lit("")),
                        F.coalesce(col("sh_record_hash"), lit(""))
                    ),
                    256
                ).alias("record_hash"),
                col("sol_created_at").alias("created_at"),
                current_timestamp().alias("etl_loaded_at")
            )
            .dropDuplicates(["sales_order_line_internal_id"])
        )

        target_cols = spark.table(gold_fact_sales).columns

        # Remove surrogate key
        merge_cols = [c for c in target_cols if c != "sales_key"]

        df_tgt = df_tgt.select(*merge_cols)

        rows_written = df_tgt.count()

        df_tgt.createOrReplaceTempView("vw_fact_sales_incremental")

        update_set = ",\n".join([
            f"target.{c} = source.{c}"
            for c in merge_cols
            if c != "sales_order_line_internal_id"
        ])

        insert_cols = ", ".join(merge_cols)
        insert_vals = ", ".join([f"source.{c}" for c in merge_cols])

        # =========================================================
        # 6. MERGE into fact
        # =========================================================
        spark.sql(f"""
            MERGE INTO {gold_fact_sales} AS target
            USING vw_fact_sales_incremental AS source
            ON target.sales_order_line_internal_id = source.sales_order_line_internal_id

            WHEN MATCHED AND target.record_hash <> source.record_hash THEN
              UPDATE SET
                {update_set}

            WHEN NOT MATCHED THEN
              INSERT ({insert_cols})
              VALUES ({insert_vals})
        """)

        records_after = spark.table(gold_fact_sales).count()

        log_layer_processed_runs_gold(source_run_ids, run_id, table_name, rows_read, rows_written, rows_rejected)

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        print(f"{gold_fact_sales}: incremental MERGE completed")
        print(f"Rows read: {rows_read}")
        print(f"Rows rejected: {rows_rejected}")
        print(f"Rows prepared for merge: {rows_written}")
        print(f"Before: {records_before}")
        print(f"After: {records_after}")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN FACT ONLY
# =========================================================
load_fact_sales_incremental()

## New Incremental for SCD2

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col, lit, current_timestamp, to_json, struct
)
import uuid
import datetime

# =========================================================
# CONFIG
# =========================================================
pipeline_name = "retail_pipeline"
layer_name = "gold"
load_type = "INCREMENTAL"
triggered_by = "manual"
source_system = "Silver_Tables"

audit_table = "retail.audit.etl_run_log"
reject_table = "retail.audit.etl_reject_log"
processed_runs_table = "retail.audit.layer_processed_runs"

batch_id = "batch_gold_fact_incremental_" + str(datetime.date.today())

# Source silver tables
silver_sales_orders = "retail.silver.ns_sales_orders"
silver_sales_order_lines = "retail.silver.ns_sales_order_lines"
silver_shipments = "retail.silver.ns_shipments"

# Target gold tables
gold_dim_customer = "retail.gold.dim_customer"
gold_dim_item = "retail.gold.dim_item"
gold_dim_date = "retail.gold.dim_date"
gold_fact_sales = "retail.gold.fact_sales"

# variable
unknown_customer_key = 350099
unknown_item_key = 7039

# =========================================================
# HELPERS
# =========================================================
def generate_run_id():
    return str(uuid.uuid4())


def start_audit(run_id, table_name):
    spark.sql(f"""
        INSERT INTO {audit_table}
        (
          run_id, batch_id, pipeline_name, layer_name, table_name, source_system,
          load_type, triggered_by, start_time, end_time, duration_seconds,
          status, rows_read, rows_written, rows_rejected, records_before,
          records_after, error_message, created_at
        )
        VALUES (
          '{run_id}', '{batch_id}', '{pipeline_name}', '{layer_name}', '{table_name}', '{source_system}',
          '{load_type}', '{triggered_by}', current_timestamp(), NULL, NULL,
          'STARTED', NULL, NULL, NULL, NULL,
          NULL, NULL, current_timestamp()
        )
    """)


def end_audit_success(run_id, rows_read, rows_written, rows_rejected, records_before, records_after):
    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'SUCCESS',
          rows_read = CAST({rows_read} AS BIGINT),
          rows_written = CAST({rows_written} AS BIGINT),
          rows_rejected = CAST({rows_rejected} AS BIGINT),
          records_before = CAST({records_before} AS BIGINT),
          records_after = CAST({records_after} AS BIGINT)
        WHERE run_id = '{run_id}'
    """)


def end_audit_fail(run_id, error_message):
    safe_error = str(error_message).replace("'", "''")[:5000]

    spark.sql(f"""
        UPDATE {audit_table}
        SET
          end_time = current_timestamp(),
          duration_seconds = unix_timestamp(current_timestamp()) - unix_timestamp(start_time),
          status = 'FAILED',
          error_message = '{safe_error}'
        WHERE run_id = '{run_id}'
    """)


def log_rejects(df_rejects, run_id, table_name):
    if df_rejects.isEmpty():
        return

    (
        df_rejects
        .withColumn("reject_id", F.expr("uuid()"))
        .withColumn("run_id", lit(run_id))
        .withColumn("batch_id", lit(batch_id))
        .withColumn("pipeline_name", lit(pipeline_name))
        .withColumn("layer_name", lit(layer_name))
        .withColumn("table_name", lit(table_name))
        .withColumn("source_system", lit(source_system))
        .withColumn("rejected_at", current_timestamp())
        .withColumn("created_at", current_timestamp())
        .select(
            "reject_id", "run_id", "batch_id", "pipeline_name", "layer_name",
            "table_name", "source_system", "source_record_id", "reject_stage",
            "reject_reason", "reject_column", "severity", "raw_payload",
            "error_code", "error_message", "rejected_at", "created_at"
        )
        .write.format("delta")
        .mode("append")
        .saveAsTable(reject_table)
    )


def get_unprocessed_silver_run_ids(source_table, gold_table_name):
    return [
        r.etl_run_id
        for r in spark.sql(f"""
            SELECT DISTINCT s.etl_run_id
            FROM {source_table} s
            LEFT ANTI JOIN {processed_runs_table} p
              ON s.etl_run_id = p.source_run_id
             AND p.table_name = '{gold_table_name}'
             AND p.source_layer = 'silver'
             AND p.target_layer = 'gold'
             AND p.status = 'SUCCESS'
            WHERE s.etl_run_id IS NOT NULL
        """).collect()
    ]


def log_layer_processed_runs_gold(source_run_ids, target_run_id, table_name, rows_read, rows_written, rows_rejected):
    for source_run_id in source_run_ids:
        spark.sql(f"""
            INSERT INTO {processed_runs_table}
            SELECT
              uuid() AS tracking_id,
              '{source_run_id}' AS source_run_id,
              '{target_run_id}' AS target_run_id,
              '{pipeline_name}' AS pipeline_name,
              'silver' AS source_layer,
              'gold' AS target_layer,
              '{table_name}' AS table_name,
              'SUCCESS' AS status,
              CAST({rows_read} AS BIGINT) AS rows_read,
              CAST({rows_written} AS BIGINT) AS rows_written,
              CAST({rows_rejected} AS BIGINT) AS rows_rejected,
              current_timestamp() AS start_time,
              current_timestamp() AS end_time,
              0 AS duration_seconds,
              NULL AS error_message,
              current_timestamp() AS created_at,
              current_timestamp() AS updated_at
        """)


# =========================================================
# FACT SALES INCREMENTAL
# =========================================================
def load_fact_sales_incremental():
    table_name = "fact_sales"
    run_id = generate_run_id()
    start_audit(run_id, table_name)

    try:
        records_before = spark.table(gold_fact_sales).count()

        sol_run_ids = get_unprocessed_silver_run_ids(silver_sales_order_lines, table_name)
        so_run_ids = get_unprocessed_silver_run_ids(silver_sales_orders, table_name)
        sh_run_ids = get_unprocessed_silver_run_ids(silver_shipments, table_name)

        source_run_ids = list(set(sol_run_ids + so_run_ids + sh_run_ids))

        if not source_run_ids:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_fact_sales}: no new silver runs")
            return

        # =========================================================
        # 1. Find affected sales orders
        # =========================================================
        affected_from_lines = (
            spark.table(silver_sales_order_lines)
            .filter(col("etl_run_id").isin(sol_run_ids))
            .select(col("sales_order_internal_id").alias("sales_order_internal_id"))
        )

        affected_from_orders = (
            spark.table(silver_sales_orders)
            .filter(col("etl_run_id").isin(so_run_ids))
            .select(col("internal_id").alias("sales_order_internal_id"))
        )

        affected_from_shipments = (
            spark.table(silver_shipments)
            .filter(col("etl_run_id").isin(sh_run_ids))
            .select(col("sales_order_internal_id").alias("sales_order_internal_id"))
        )

        df_affected_orders = (
            affected_from_lines
            .union(affected_from_orders)
            .union(affected_from_shipments)
            .filter(col("sales_order_internal_id").isNotNull())
            .dropDuplicates()
        )

        affected_order_count = df_affected_orders.count()

        if affected_order_count == 0:
            end_audit_success(run_id, 0, 0, 0, records_before, records_before)
            print(f"{gold_fact_sales}: no affected orders found")
            return

        # =========================================================
        # 2. Read affected sales order lines
        # =========================================================
        df_sol = (
            spark.table(silver_sales_order_lines)
            .join(
                df_affected_orders,
                on="sales_order_internal_id",
                how="inner"
            )
            .select(
                col("internal_id").alias("sol_internal_id"),
                col("sales_order_internal_id").alias("sol_sales_order_internal_id"),
                col("line_num").alias("sol_line_num"),
                col("item_internal_id").alias("sol_item_internal_id"),
                col("quantity").alias("sol_quantity"),
                col("unit_rate").alias("sol_unit_rate"),
                col("line_amount").alias("sol_line_amount"),
                col("record_hash").alias("sol_record_hash"),
                col("created_at").alias("sol_created_at")
            )
        )

        df_so = (
            spark.table(silver_sales_orders)
            .select(
                col("internal_id").alias("so_internal_id"),
                col("customer_internal_id").alias("so_customer_internal_id"),
                col("order_date").alias("so_order_date"),
                col("order_status").alias("so_order_status"),
                col("sales_channel").alias("so_sales_channel"),
                col("payment_status").alias("so_payment_status"),
                col("currency_code").alias("so_currency_code"),
                col("integration_source").alias("so_integration_source"),
                col("order_total").alias("so_order_total"),
                col("source_system").alias("so_source_system"),
                col("etl_run_id").alias("so_etl_run_id"),
                col("record_hash").alias("so_record_hash")
            )
        )

        df_sh = (
            spark.table(silver_shipments)
            .select(
                col("internal_id").alias("sh_internal_id"),
                col("sales_order_internal_id").alias("sh_sales_order_internal_id"),
                col("shipment_date").alias("sh_shipment_date"),
                col("delivery_date").alias("sh_delivery_date"),
                col("shipment_status").alias("sh_shipment_status"),
                col("warehouse_name").alias("sh_warehouse_name"),
                col("tracking_number").alias("sh_tracking_number"),
                col("record_hash").alias("sh_record_hash")
            )
        )

        df_joined = (
            df_sol
            .join(
                df_so,
                col("sol_sales_order_internal_id") == col("so_internal_id"),
                "inner"
            )
            .join(
                df_sh,
                col("so_internal_id") == col("sh_sales_order_internal_id"),
                "left"
            )
        )

        rows_read = df_joined.count()

        # =========================================================
        # 3. Dimension lookups
        # =========================================================
        df_customer_lkp = (
            spark.table(gold_dim_customer)
            .select(
                col("customer_key"),
                col("customer_internal_id").cast("int").alias("lkp_customer_internal_id"),
                col("effective_start_date").alias("cust_effective_start_date"),
                col("effective_end_date").alias("cust_effective_end_date")
            )
        )

        df_item_lkp = (
            spark.table(gold_dim_item)
            .select(
                col("item_key"),
                col("item_internal_id").cast("int").alias("lkp_item_internal_id"),
                col("effective_start_date").alias("item_effective_start_date"),
                col("effective_end_date").alias("item_effective_end_date")
            )
        )

        df_date_lkp = spark.table(gold_dim_date).select(
            col("date_key"),
            col("full_date").alias("lkp_full_date")
        )

        df_fact_pre = (
            df_joined
            .join(
                df_customer_lkp,
                (col("so_customer_internal_id").cast("int") == col("lkp_customer_internal_id"))
                & (F.to_timestamp(col("so_order_date")) >= col("cust_effective_start_date"))
                & (
                    col("cust_effective_end_date").isNull()
                    | (F.to_timestamp(col("so_order_date")) < col("cust_effective_end_date"))
                ),
                "left"
            )
            .join(
                df_item_lkp,
                (col("sol_item_internal_id").cast("int") == col("lkp_item_internal_id"))
                & (F.to_timestamp(col("so_order_date")) >= col("item_effective_start_date"))
                & (
                    col("item_effective_end_date").isNull()
                    | (F.to_timestamp(col("so_order_date")) < col("item_effective_end_date"))
                ),
                "left"
            )
            .join(
                df_date_lkp,
                F.to_date(col("so_order_date")) == col("lkp_full_date"),
                "left"
            )
        )

        # =========================================================
        # 4. Handle failed lookups using Unknown Dimension Strategy
        # =========================================================

        # 🔹 Step 4.1 — Count missing lookups BEFORE fallback (for audit)
        unknown_lookup_rows = df_fact_pre.filter(
            col("customer_key").isNull() |
            col("item_key").isNull() |
            col("date_key").isNull()
        ).count()

        # 🔹 Step 4.2 — Fetch unknown dimension keys
        unknown_customer_key = spark.sql("""
            SELECT customer_key
            FROM retail.gold.dim_customer
            WHERE customer_internal_id = -1
            AND is_current = true
        """).collect()[0]["customer_key"]

        unknown_item_key = spark.sql("""
            SELECT item_key
            FROM retail.gold.dim_item
            WHERE item_internal_id = -1
            AND is_current = true
        """).collect()[0]["item_key"]

        unknown_date_key = -1

        # 🔹 Step 4.3 — Apply fallback (NO REJECTION)
        df_fact_pre = (
            df_fact_pre
            .withColumn(
                "customer_key",
                F.coalesce(col("customer_key"), lit(unknown_customer_key).cast("bigint"))
            )
            .withColumn(
                "item_key",
                F.coalesce(col("item_key"), lit(unknown_item_key).cast("bigint"))
            )
            .withColumn(
                "date_key",
                F.coalesce(col("date_key"), lit(unknown_date_key).cast("int"))
            )
        )

        # 🔹 Step 4.4 — No rejection anymore
        rows_rejected = 0
        df_valid = df_fact_pre

        # =========================================================
        # 5. Prepare fact table rows
        # =========================================================
        df_tgt = (
            df_valid
            .select(
                col("sol_internal_id").alias("sales_order_line_internal_id"),
                col("sol_sales_order_internal_id").alias("sales_order_internal_id"),
                col("sol_line_num").alias("sales_order_line_num"),
                col("date_key").alias("order_date_key"),
                col("customer_key"),
                col("item_key"),
                col("sol_quantity").alias("quantity"),
                col("sol_unit_rate").alias("unit_rate"),
                col("sol_line_amount").alias("line_amount"),
                col("so_order_total").alias("order_total"),
                col("so_order_status").alias("order_status"),
                col("so_sales_channel").alias("sales_channel"),
                col("so_payment_status").alias("payment_status"),
                col("so_currency_code").alias("currency_code"),
                col("so_integration_source").alias("integration_source"),
                col("sh_shipment_date").alias("shipment_date"),
                col("sh_delivery_date").alias("delivery_date"),
                col("sh_shipment_status").alias("shipment_status"),
                col("sh_warehouse_name").alias("warehouse_name"),
                col("sh_tracking_number").alias("tracking_number"),
                col("so_source_system").alias("source_system"),
                col("so_etl_run_id").alias("etl_run_id"),
                (col("customer_key") == lit(unknown_customer_key)).alias("has_unknown_customer"),
                (col("item_key") == lit(unknown_item_key)).alias("has_unknown_item"),
                (col("date_key") == lit(-1)).alias("has_unknown_date"),
                (col("sol_quantity") < 0).alias("has_negative_quantity"),
                (
                    (col("sol_unit_rate") < 0) |
                    (col("sol_line_amount") < 0) |
                    (col("so_order_total") < 0)
                ).alias("has_negative_amount"),
                F.when(
                    (col("customer_key") == lit(unknown_customer_key)) |
                    (col("item_key") == lit(unknown_item_key)) |
                    (col("date_key") == lit(-1)),
                    lit("UNKNOWN_DIMENSION")
                ).when(
                    (col("sol_quantity") < 0) |
                    (col("sol_unit_rate") < 0) |
                    (col("sol_line_amount") < 0) |
                    (col("so_order_total") < 0),
                    lit("INVALID_DATA")
                ).otherwise(lit("VALID")).alias("dq_status"),
                F.concat_ws(
                    ", ",
                    F.when(col("customer_key") == lit(unknown_customer_key), lit("UNKNOWN_CUSTOMER")),
                    F.when(col("item_key") == lit(unknown_item_key), lit("UNKNOWN_ITEM")),
                    F.when(col("date_key") == lit(-1), lit("UNKNOWN_DATE")),
                    F.when(col("sol_quantity") < 0, lit("NEGATIVE_QUANTITY")),
                    F.when(
                        (col("sol_unit_rate") < 0) |
                        (col("sol_line_amount") < 0) |
                        (col("so_order_total") < 0),
                        lit("NEGATIVE_AMOUNT")
                    )
                ).alias("dq_reason"),
                (
                    ~(
                        (col("customer_key") == lit(unknown_customer_key)) |
                        (col("item_key") == lit(unknown_item_key)) |
                        (col("date_key") == lit(-1)) |
                        (col("sol_quantity") < 0) |
                        (col("sol_unit_rate") < 0) |
                        (col("sol_line_amount") < 0) |
                        (col("so_order_total") < 0)
                    )
                ).alias("is_valid"),
                current_timestamp().alias("dq_checked_at"),
                F.sha2(
                    F.concat_ws(
                        "||",
                        F.coalesce(col("sol_record_hash"), lit("")),
                        F.coalesce(col("so_record_hash"), lit("")),
                        F.coalesce(col("sh_record_hash"), lit("")),
                        F.coalesce(col("customer_key").cast("string"), lit("")),
                        F.coalesce(col("item_key").cast("string"), lit("")),
                        F.coalesce(col("date_key").cast("string"), lit(""))
                    ),
                    256
                ).alias("record_hash"),
                col("sol_created_at").alias("created_at"),
                current_timestamp().alias("etl_loaded_at")
            )
            .dropDuplicates(["sales_order_line_internal_id"])
        )

        target_cols = spark.table(gold_fact_sales).columns

        # Remove surrogate key
        merge_cols = [c for c in target_cols if c != "sales_key"]

        df_tgt = df_tgt.select(*merge_cols)

        rows_written = df_tgt.count()

        df_tgt.createOrReplaceTempView("vw_fact_sales_incremental")

        update_set = ",\n".join([
            f"target.{c} = source.{c}"
            for c in merge_cols
            if c != "sales_order_line_internal_id"
        ])

        insert_cols = ", ".join(merge_cols)
        insert_vals = ", ".join([f"source.{c}" for c in merge_cols])

        # =========================================================
        # 6. MERGE into fact
        # =========================================================
        spark.sql(f"""
            MERGE INTO {gold_fact_sales} AS target
            USING vw_fact_sales_incremental AS source
            ON target.sales_order_line_internal_id = source.sales_order_line_internal_id

            WHEN MATCHED AND target.record_hash <> source.record_hash THEN
              UPDATE SET
                {update_set}

            WHEN NOT MATCHED THEN
              INSERT ({insert_cols})
              VALUES ({insert_vals})
        """)

        records_after = spark.table(gold_fact_sales).count()

        log_layer_processed_runs_gold(source_run_ids, run_id, table_name, rows_read, rows_written, rows_rejected)

        end_audit_success(
            run_id=run_id,
            rows_read=rows_read,
            rows_written=rows_written,
            rows_rejected=rows_rejected,
            records_before=records_before,
            records_after=records_after
        )

        # =========================================================
        # 7. DQ METRICS INSERT
        # =========================================================

        spark.sql(f"""
        INSERT INTO retail.monitoring.dq_metrics
        SELECT
        '{run_id}' AS run_id,
        '{batch_id}' AS batch_id,
        'fact_sales' AS table_name,
        'gold' AS layer_name,

        COUNT(*) AS total_rows,
        SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) AS valid_rows,
        SUM(CASE WHEN NOT is_valid THEN 1 ELSE 0 END) AS invalid_rows,
        SUM(CASE WHEN dq_status = 'UNKNOWN_DIMENSION' THEN 1 ELSE 0 END),

        SUM(line_amount) AS total_revenue,
        SUM(CASE WHEN is_valid THEN line_amount ELSE 0 END),
        SUM(CASE WHEN NOT is_valid THEN line_amount ELSE 0 END),

        ROUND(
            100.0 * SUM(CASE WHEN is_valid THEN 1 ELSE 0 END) / COUNT(*),
            4
        ) AS dq_score,

        current_timestamp()
        FROM retail.gold.fact_sales
        """)

        print(f"{gold_fact_sales}: incremental MERGE completed")
        print(f"Rows read: {rows_read}")
        print(f"Rows rejected: {rows_rejected}")
        print(f"Rows prepared for merge: {rows_written}")
        print(f"Before: {records_before}")
        print(f"After: {records_after}")

    except Exception as e:
        end_audit_fail(run_id, str(e))
        raise


# =========================================================
# RUN FACT ONLY
# =========================================================
load_fact_sales_incremental()